In [1]:
from PIL import Image
import cv2
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
PASTA_IMG = r'/home/joaoinacio/bootcamp-machine-learning/data/raw'
IMG_PROCESSADAS = r'/home/joaoinacio/bootcamp-machine-learning/data/processed'
PASTA_RAIZ = "/home/joaoinacio/bootcamp-machine-learning"

In [3]:
def gerarDataset(diretorio_imagens, nome_csv):
    lista_dados = []
    diretorio_dataset = os.path.normpath(diretorio_imagens)
    root_projeto = os.path.dirname(os.path.dirname(diretorio_dataset))

    for root, dirs, files in os.walk(diretorio_imagens):
        for file in files:
            if file.lower().endswith(
                (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp")
            ):
                caminho_completo = os.path.join(root, file)
                caminho_relativo = os.path.relpath(caminho_completo, root_projeto)
                caminho_relativo = caminho_relativo.replace("\\", "/")
                subtipo = os.path.basename(root)
                pasta_pai = os.path.dirname(root)
                classe_principal = os.path.basename(pasta_pai)

                label_composto = f"{classe_principal}/{subtipo}"
                lista_dados.append({
                    "caminho_completo": caminho_relativo,
                    "nome_arquivo": file,
                    "classe": classe_principal,
                    "subtipo": subtipo,
                    "label_final": label_composto,
                })
    if lista_dados:
        df = pd.DataFrame(lista_dados)
        df = df.sort_values(by="label_final")
        df.to_csv(f"{IMG_PROCESSADAS}/{nome_csv}", index=False)
        print(f"Dataset criado na pasta: {IMG_PROCESSADAS}")

In [4]:
gerarDataset(PASTA_IMG, "dados.csv")

Dataset criado na pasta: /home/joaoinacio/bootcamp-machine-learning/data/processed


In [5]:
df = pd.read_csv('../data/processed/dados.csv')
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default


In [ ]:
df['caminho_completo']

In [6]:
def verificaResolucao(caminho, root_projeto):
    tamanho = []
    for imgs in caminho:
        caminho_absoluto = os.path.join(root_projeto, imgs)
        img = Image.open(caminho_absoluto)
        tamanho.append(img.size)
    return tamanho

In [7]:
resolucao = verificaResolucao(df['caminho_completo'], PASTA_RAIZ)
print(resolucao)
print(np.min(resolucao))
print(np.max(resolucao))

[(256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (256, 256), (25

In [8]:
df[['largura', 'altura']] = pd.DataFrame(resolucao, index=df.index)
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256


In [10]:
def calcular_variancia_laplaciano(img_path, root_projeto):
    variancias = []

    for imgs in img_path:
        caminho_absoluto = os.path.join(root_projeto, imgs)
        image = cv2.imread(caminho_absoluto, cv2.IMREAD_GRAYSCALE)
        if image is None:
            return None
        laplacian = cv2.Laplacian(image, cv2.CV_64F)

        variance = laplacian.var()
        variancias.append(variance)

    return variancias

In [11]:
score = calcular_variancia_laplaciano(df['caminho_completo'], PASTA_RAIZ)
print(score)
print(np.min(score))
print(np.max(score))

[np.float64(921.7503960495815), np.float64(2528.4307170053944), np.float64(1063.6627959811594), np.float64(2631.672543144785), np.float64(1606.7586364746094), np.float64(1453.7401001711842), np.float64(161.6772002147045), np.float64(354.43011450767517), np.float64(2776.565609975718), np.float64(158.85977172851562), np.float64(1732.3688354156911), np.float64(415.9328918457031), np.float64(730.2503559431061), np.float64(477.2726352547761), np.float64(3081.899627685547), np.float64(1559.193448599428), np.float64(1205.8013762587216), np.float64(747.154295979999), np.float64(92.5283203125), np.float64(1455.3535422077402), np.float64(1459.6157348519191), np.float64(5189.415307244053), np.float64(2081.1029801033437), np.float64(2575.4197614258155), np.float64(1353.1218383675441), np.float64(2107.860035573831), np.float64(174.87710177805275), np.float64(2691.805142108118), np.float64(8336.460083007812), np.float64(2616.278915403178), np.float64(542.728132473072), np.float64(396.1403503417969),

In [12]:
df['Variancia_laplaciana'] = pd.DataFrame(score, df.index)
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura,Variancia_laplaciana
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256,921.750396
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256,2528.430717
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256,1063.662796
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256,2631.672543
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256,1606.758636


In [ ]:
minin = []
for scr in score:
    if scr < 100:
        minin.append(scr)

print(np.size(minin))

In [13]:
def verificaAspectRatio(caminho, root_projeto):
    ratio = []
    for imgs in caminho:
        caminho_absoluto = os.path.join(root_projeto, imgs)
        img = Image.open(caminho_absoluto)
        largura, altura = img.size
        if altura > 0:
            razao = largura / altura
            ratio.append(razao)
    return ratio

In [14]:
asp_ratio = verificaAspectRatio(df['caminho_completo'], PASTA_RAIZ)
print(asp_ratio)
print(np.min(asp_ratio))
print(np.max(asp_ratio))

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,

In [15]:
df['Aspect_Ratio'] = pd.DataFrame(asp_ratio, df.index)
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura,Variancia_laplaciana,Aspect_Ratio
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256,921.750396,1.0
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256,2528.430717,1.0
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256,1063.662796,1.0
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256,2631.672543,1.0
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256,1606.758636,1.0


In [22]:
def canaisCor(caminho, root_projeto):
    cores = []
    for imgs in caminho:
        caminho_absoluto = os.path.join(root_projeto, imgs)
        img = Image.open(caminho_absoluto)
        cores.append(img.mode)
    return cores

In [ ]:
canais = canaisCor(df['caminho_completo'], PASTA_RAIZ)
print(canais[0])

RGB


In [24]:
df['canais_cor'] = pd.DataFrame(canais, df.index)
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura,Variancia_laplaciana,Aspect_Ratio,canais_cor
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256,921.750396,1.0,RGB
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256,2528.430717,1.0,RGB
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256,1063.662796,1.0,RGB
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256,2631.672543,1.0,RGB
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256,1606.758636,1.0,RGB


In [ ]:
def tamanhoArquivo(caminho, root_projeto):
    tamanho = []
    for imgs in caminho:
        caminho_absoluto = os.path.join(root_projeto, imgs)
        if os.path.exists(caminho_absoluto):
            tamanho_bytes = os.path.getsize(caminho_absoluto)
            tamanho_kb = tamanho_bytes / 1024
            tamanho.append(tamanho_kb)
    return tamanho

In [27]:
tamanho_arquivo = tamanhoArquivo(df['caminho_completo'], PASTA_RAIZ)
print(tamanho_arquivo)

[50.974609375, 77.431640625, 55.3642578125, 75.859375, 53.3212890625, 54.7080078125, 25.9033203125, 35.904296875, 79.1396484375, 9.9765625, 82.00390625, 34.7646484375, 41.01171875, 58.9970703125, 52.74609375, 63.0458984375, 56.318359375, 43.9814453125, 19.4755859375, 40.642578125, 84.93359375, 137.3173828125, 79.1376953125, 74.53125, 27.091796875, 89.9033203125, 26.9599609375, 103.3779296875, 40.0048828125, 56.9404296875, 43.392578125, 70.2861328125, 38.388671875, 49.365234375, 46.861328125, 87.27734375, 76.5, 53.0234375, 24.40234375, 66.0927734375, 41.5673828125, 52.9013671875, 49.837890625, 45.1845703125, 37.7529296875, 68.45703125, 60.3115234375, 24.328125, 57.8037109375, 50.912109375, 73.1875, 45.0537109375, 29.2099609375, 70.01953125, 80.359375, 24.40234375, 40.4970703125, 66.3857421875, 9.21875, 33.1884765625, 51.884765625, 48.0244140625, 86.025390625, 44.7470703125, 76.78125, 33.5576171875, 61.568359375, 75.4384765625, 29.341796875, 91.5673828125, 35.6279296875, 70.939453125, 14

In [30]:
print(round(tamanho_arquivo[0], 2))

50.97


In [28]:
df['tamanho_kb'] = pd.DataFrame(tamanho_arquivo, df.index)
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura,Variancia_laplaciana,Aspect_Ratio,canais_cor,tamanho_kb
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256,921.750396,1.0,RGB,50.974609
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256,2528.430717,1.0,RGB,77.431641
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256,1063.662796,1.0,RGB,55.364258
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256,2631.672543,1.0,RGB,75.859375
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256,1606.758636,1.0,RGB,53.321289


In [31]:
df['tamanho_kb'] = df['tamanho_kb'].apply(lambda x: round(x, 2))
df.head()

,caminho_completo,nome_arquivo,classe,subtipo,label_final,largura,altura,Variancia_laplaciana,Aspect_Ratio,canais_cor,tamanho_kb
0,data/raw/aerosol_cans/default/Image_1.png,Image_1.png,aerosol_cans,default,aerosol_cans/default,256,256,921.750396,1.0,RGB,50.97
1,data/raw/aerosol_cans/default/Image_10.png,Image_10.png,aerosol_cans,default,aerosol_cans/default,256,256,2528.430717,1.0,RGB,77.43
2,data/raw/aerosol_cans/default/Image_100.png,Image_100.png,aerosol_cans,default,aerosol_cans/default,256,256,1063.662796,1.0,RGB,55.36
3,data/raw/aerosol_cans/default/Image_101.png,Image_101.png,aerosol_cans,default,aerosol_cans/default,256,256,2631.672543,1.0,RGB,75.86
4,data/raw/aerosol_cans/default/Image_102.png,Image_102.png,aerosol_cans,default,aerosol_cans/default,256,256,1606.758636,1.0,RGB,53.32
